# Movies Recomendation System

 we want to build a model that takes user ID and then output a list of the top 5 movies that the user is likely to enjoy.
 
 **How does it work?..**
 
 we train the model to predict the rating that the user would give to all the movies he/she didn't watch (assuming that the movies     he/she didn't rate is unwatched yet), and then we sort the the list of predicted ratings so we can take the top 5 and output the      short list.



## 1. Data Loading & Exploration

In [1]:
import numpy as np 
import pandas as pd 

In [2]:
# Load ratings data from u.data
ratings = pd.read_csv("/kaggle/input/datasets/noureleman/movielens100k/ml-100k/u.data")
ratings

,196\t242\t3\t881250949
0,186\t302\t3\t891717742
1,22\t377\t1\t878887116
2,244\t51\t2\t880606923
3,166\t346\t1\t886397596
4,298\t474\t4\t884182806
...,...
99994,880\t476\t3\t880175444
99995,716\t204\t5\t879795543
99996,276\t1090\t1\t874795795
99997,13\t225\t2\t882399156


In [3]:
ratings = pd.read_csv("/kaggle/input/datasets/noureleman/movielens100k/ml-100k/u.data", sep="\t", names=["user_id", "movie_id", "rating", "timestamp"])
ratings

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [4]:
# Load movie metadata from u.item (adjust separator and encoding if needed)
movies = pd.read_csv("/kaggle/input/datasets/noureleman/movielens100k/ml-100k/u.item", sep="|", encoding="latin-1", 
                     names=["movie_id", "title", "release_date", "video_release_date", "IMDb_URL", "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"])
movies

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,NaN,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Load user data if needed
users = pd.read_csv("/kaggle/input/datasets/noureleman/movielens100k/ml-100k/u.user", sep="|", names=["user_id", "age", "gender", "occupation", "zip_code"])
users

,user_id,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213
...,...,...,...,...,...
938,939,26,F,student,33319
939,940,32,M,administrator,02215
940,941,20,M,student,97229
941,942,48,F,librarian,78209


#### In this notebook we are going to use *collaborative-filtering* approach, so in a basic model we only need [movie ID, user ID, rating, title(for interpretation, not for modeling)] , so here we're going to craete a DF to include only the needed columns. 

In [6]:
ratings.drop('timestamp', axis=1, inplace=True)  # we won't need 'timesatamp'
#normalize ratings
ratings['rating'] = (ratings['rating'] - 1.0) / 4.0  # Scale to [0, 1]
ratings

,user_id,movie_id,rating
0,196,242,0.50
1,186,302,0.50
2,22,377,0.00
3,244,51,0.25
4,166,346,0.00
...,...,...,...
99995,880,476,0.50
99996,716,204,1.00
99997,276,1090,0.00
99998,13,225,0.25


In [7]:
data = pd.merge(ratings, movies[['movie_id', 'title']], on='movie_id')
data

,user_id,movie_id,rating,title
0,196,242,0.50,Kolya (1996)
1,186,302,0.50,L.A. Confidential (1997)
2,22,377,0.00,Heavyweights (1994)
3,244,51,0.25,Legends of the Fall (1994)
4,166,346,0.00,Jackie Brown (1997)
...,...,...,...,...
99995,880,476,0.50,"First Wives Club, The (1996)"
99996,716,204,1.00,Back to the Future (1985)
99997,276,1090,0.00,Sliver (1993)
99998,13,225,0.25,101 Dalmatians (1996)


In [8]:
# Extracting user IDs, movie IDs, and ratings
user_ids = data['user_id'].values  # Convert to numpy array
movie_ids = data['movie_id'].values  
ratings = data['rating'].values    

## 2. Data processing 

In [9]:
data.duplicated().sum()

0

In [10]:
data.isnull().sum()

user_id     0
movie_id    0
rating      0
title       0
dtype: int64

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   user_id   100000 non-null  int64  
 1   movie_id  100000 non-null  int64  
 2   rating    100000 non-null  float64
 3   title     100000 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 3.1+ MB


In [12]:
data.describe()

,user_id,movie_id,rating
count,100000.00000,100000.000000,100000.000000
mean,462.48475,425.530130,0.632465
std,266.61442,330.798356,0.281418
min,1.00000,1.000000,0.000000
25%,254.00000,175.000000,0.500000
50%,447.00000,322.000000,0.750000
75%,682.00000,631.000000,0.750000
max,943.00000,1682.000000,1.000000


In [13]:
data.rating.value_counts()

rating
0.75    34174
0.50    27145
1.00    21201
0.25    11370
0.00     6110
Name: count, dtype: int64

## 3. Preparing the Data for Collaborative Filtering

To achieve that, here the steps we are going to follow:
1. create a user-item matrix
2. convert the Matrix to a Tensor (for PyTorch)

In [14]:
print(data.duplicated(subset=['user_id', 'title']).sum())  # Check duplicate user-movie ratings


307


In [15]:
data = data.drop_duplicates(subset=['user_id', 'title'])

In [16]:
print(data['title'].nunique())  # Check unique movie count


1664


In [17]:
# creating user-item matrix & replacing null values with 0
user_movie_matrix = data.pivot(index='user_id', columns='title', values='rating').fillna(0)
user_movie_matrix

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.25,1.0,0.00,0.00,0.5,0.75,0.0,0.0,...,0.0,0.0,0.0,1.00,0.50,0.0,0.0,0.0,0.75,0.0
2,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0
3,0.0,0.0,0.00,0.0,0.25,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0
4,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0
5,0.0,0.0,0.25,0.0,0.00,0.00,0.0,0.75,0.0,0.0,...,0.0,0.0,0.0,0.75,0.00,0.0,0.0,0.0,0.75,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0
940,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0
941,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.00,0.0


## 4. Model Building 

In [18]:
import torch
import random

In [19]:
#Converting the matrix to a tensor
user_movie_tensor = torch.tensor(user_movie_matrix.values, dtype=torch.float32)

In [20]:
#Checking the tensor shape
user_movie_tensor.shape

torch.Size([943, 1664])

In [21]:
# Calculating the number of unique users and movies so we can use it later
num_users, num_movies = user_movie_tensor.shape
num_users, num_movies

(943, 1664)

In [22]:
# turning user_ids, movie_ids, ratings ino tensors
user_ids = torch.tensor(user_ids, dtype=torch.long)
movie_ids = torch.tensor(movie_ids, dtype=torch.long)
ratings = torch.tensor(ratings, dtype=torch.float)

## Split Data into Train/Test Sets

In [23]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Split indices
train_indices, test_indices = train_test_split(range(len(ratings)), test_size=0.2, random_state=42)

# Create train/test datasets
train_dataset = TensorDataset(user_ids[train_indices], movie_ids[train_indices], ratings[train_indices])
test_dataset = TensorDataset(user_ids[test_indices], movie_ids[test_indices], ratings[test_indices])

# Create DataLoaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### 4.1 Defining the model

In [24]:
import torch.nn as nn

class CollaborativeFiltering(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super(CollaborativeFiltering, self).__init__()
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.user_embeddings.weight, std=0.01)
        nn.init.normal_(self.item_embeddings.weight, std=0.01)
        nn.init.normal_(self.user_bias.weight, std=0.01)
        nn.init.normal_(self.item_bias.weight, std=0.01)

    def forward(self, user_ids, item_ids):
        user_embed = self.user_embeddings(user_ids)
        item_embed = self.item_embeddings(item_ids)
        user_bias = self.user_bias(user_ids).squeeze()
        item_bias = self.item_bias(item_ids).squeeze()
        
        # Calculate raw score
        raw_score = (user_embed * item_embed).sum(dim=1) + user_bias + item_bias
        
        # Force the output to be between 0 and 1
        return torch.sigmoid(raw_score)

# 5. Model Training

## 5.1 Defining the Loss Function and Optimizer

In [25]:
import torch.optim as optim

num_users = user_ids.max().item() + 1
num_items = movie_ids.max().item() + 1
embedding_dim = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CollaborativeFiltering(num_users, num_items, embedding_dim).to(device)

loss_fn = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)


## 5.3 Training Loop

In [26]:
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch in train_dataloader:
        user_batch, item_batch, rating_batch = batch
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        rating_batch = rating_batch.to(device)
        
        predictions = model(user_batch, item_batch)
        loss = loss_fn(predictions, rating_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss / len(train_dataloader):.4f}")

Epoch [1/20], Loss: 0.0739
Epoch [2/20], Loss: 0.0564
Epoch [3/20], Loss: 0.0526
Epoch [4/20], Loss: 0.0504
Epoch [5/20], Loss: 0.0487
Epoch [6/20], Loss: 0.0470
Epoch [7/20], Loss: 0.0452
Epoch [8/20], Loss: 0.0433
Epoch [9/20], Loss: 0.0412
Epoch [10/20], Loss: 0.0393
Epoch [11/20], Loss: 0.0373
Epoch [12/20], Loss: 0.0356
Epoch [13/20], Loss: 0.0339
Epoch [14/20], Loss: 0.0325
Epoch [15/20], Loss: 0.0312
Epoch [16/20], Loss: 0.0300
Epoch [17/20], Loss: 0.0290
Epoch [18/20], Loss: 0.0281
Epoch [19/20], Loss: 0.0273
Epoch [20/20], Loss: 0.0266


## Evaluate the Model

In [27]:
model.eval()
test_loss = 0.0
with torch.no_grad():
    for batch in test_dataloader:
        user_batch, item_batch, rating_batch = batch
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        rating_batch = rating_batch.to(device)
        
        predictions = model(user_batch, item_batch)
        test_loss += loss_fn(predictions, rating_batch).item()

test_loss /= len(test_dataloader)
rmse = torch.sqrt(torch.tensor(test_loss)).item()
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test RMSE: {rmse:.4f}")

Test Loss (MSE): 0.0518
Test RMSE: 0.2277


In [28]:
def recommend_top_k(model, user_id, all_movie_ids, rated_movie_ids, device='cpu', k=5):
    # Get movies the user HAS NOT rated
    unseen_movie_ids = list(set(all_movie_ids) - set(rated_movie_ids))
    unseen_movie_ids = torch.tensor(unseen_movie_ids, dtype=torch.long).to(device)
    
    # Predict ratings for unseen movies
    user_ids = torch.full_like(unseen_movie_ids, user_id).to(device)
    with torch.no_grad():
        predictions = model(user_ids, unseen_movie_ids)
    
    # Get top k
    _, top_indices = torch.topk(predictions, k=k)
    return unseen_movie_ids[top_indices].cpu().numpy()

In [29]:
#example usage
user_id = 0
rated_movie_ids = movie_ids[user_ids == user_id].tolist()
top_5_movies = recommend_top_k(model, user_id, range(num_items), rated_movie_ids, device)
print(f"Top 5 movies for user {user_id}: {top_5_movies}")

Top 5 movies for user 0: [ 64 318  50  12 483]


## Testing

In [30]:
from sklearn.model_selection import train_test_split

# Split indices (assuming user_ids, movie_ids, ratings are your full dataset)
indices = range(len(ratings))
train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42)

# Create train/test datasets
train_dataset = TensorDataset(
    user_ids[train_indices], 
    movie_ids[train_indices], 
    ratings[train_indices]
)
test_dataset = TensorDataset(
    user_ids[test_indices], 
    movie_ids[test_indices], 
    ratings[test_indices]
)

# Create DataLoaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [31]:
import numpy as np

test_loss = 0.0
all_predictions = []
all_truth = []

with torch.no_grad():
    for batch in test_dataloader:
        user_batch, movie_batch, rating_batch = batch
        user_batch = user_batch.to(device)
        movie_batch = movie_batch.to(device)
        rating_batch = rating_batch.to(device)
        
        predictions = model(user_batch, movie_batch)
        loss = loss_fn(predictions, rating_batch)
        test_loss += loss.item()
        
        # Collect predictions and true ratings
        all_predictions.extend(predictions.cpu().numpy())
        all_truth.extend(rating_batch.cpu().numpy())

# Calculate average loss and RMSE
test_loss /= len(test_dataloader)
rmse = np.sqrt(test_loss)
print(f"Test Loss (MSE): {test_loss:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# If you normalized ratings, rescale predictions for interpretation
# Example: if ratings were scaled to [0, 1], reverse the scaling
# all_predictions = [p * 4.0 + 1.0 for p in all_predictions]
# all_truth = [t * 4.0 + 1.0 for t in all_truth]

Test Loss (MSE): 0.0518
Test RMSE: 0.2277


In [32]:
torch.save(model.state_dict(), 'movie_recommender.pth')